# Quy Trình Feature Engineering - Shopping Mall Customer Segmentation

Mục tiêu của notebook này là thực hiện Kỹ nghệ đặc trưng (Feature Engineering) trên bộ dữ liệu `Shopping Mall Customer Segmentation Data .csv` theo các bước trong `feature_engineering_guide.md` để chuẩn bị dữ liệu cho thuật toán phân cụm.

## Bước 1. Xử Lý Giá Trị Thiếu (Handling Missing Values)

Kiểm tra và xử lý các giá trị khuyết thiếu trong dữ liệu.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Đọc dữ liệu
data_path = '../data/raw/Shopping Mall Customer Segmentation Data .csv'
df = pd.read_csv(data_path)

# Kiểm tra dữ liệu thiếu
print("Số lượng giá trị khuyết thiếu:")
print(df.isnull().sum())
print("\n=> Nhận xét: Bộ dữ liệu hoàn toàn sạch, không có giá trị khuyết thiếu. Không cần xử lý điền khuyết (Imputation).")

Số lượng giá trị khuyết thiếu:
Customer ID       0
Age               0
Gender            0
Annual Income     0
Spending Score    0
dtype: int64

=> Nhận xét: Bộ dữ liệu hoàn toàn sạch, không có giá trị khuyết thiếu. Không cần xử lý điền khuyết (Imputation).


## Bước 2. Xử Lý Giá Trị Ngoại Lệ (Handling Outliers)

Kiểm tra và xử lý các giá trị ngoại lệ.

In [2]:
# Kiểm tra outliers dựa trên IQR
def check_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers)

for col in ['Age', 'Annual Income', 'Spending Score']:
    print(f"Số lượng ngoại lệ cột {col}: {check_outliers(df[col])}")
print("\n=> Nhận xét: Không có điểm ngoại lệ nào vượt quá ngưỡng 1.5 * IQR. Dữ liệu cực kỳ đồng đều, không cần cắt tỉa hay giới hạn biên (Winsorization/Capping).")

Số lượng ngoại lệ cột Age: 0
Số lượng ngoại lệ cột Annual Income: 0
Số lượng ngoại lệ cột Spending Score: 0

=> Nhận xét: Không có điểm ngoại lệ nào vượt quá ngưỡng 1.5 * IQR. Dữ liệu cực kỳ đồng đều, không cần cắt tỉa hay giới hạn biên (Winsorization/Capping).


## Bước 3. Mã Hóa Biến Phân Loại (Categorical Encoding)

Chuyển đổi đặc trưng giới tính `Gender` (Male, Female) thành số nguyên nhị phân (1 và 0).

In [3]:
# Tạo cột mới Gender_encoded: Male -> 1, Female -> 0
df['Gender_encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})
print("Tỷ lệ phân bố của giới tính sau mã hóa:")
print(df['Gender_encoded'].value_counts())
df.head()

Tỷ lệ phân bố của giới tính sau mã hóa:
Gender_encoded
1    7595
0    7484
Name: count, dtype: int64


,Customer ID,Age,Gender,Annual Income,Spending Score,Gender_encoded
0,d410ea53-6661-42a9-ad3a-f554b05fd2a7,30,Male,151479,89,1
1,1770b26f-493f-46b6-837f-4237fb5a314e,58,Female,185088,95,0
2,e81aa8eb-1767-4b77-87ce-1620dc732c5e,62,Female,70912,76,0
3,9795712a-ad19-47bf-8886-4f997d6046e3,23,Male,55460,57,1
4,64139426-2226-4cd6-bf09-91bce4b4db5e,24,Male,153752,76,1


## Bước 4. Biến Đổi & Chuẩn Hóa Biến Số (Numerical Transformation & Scaling)

Chuẩn hóa thang đo của `Age`, `Annual Income`, và `Spending Score` bằng `StandardScaler` để các thuật toán phân cụm hoạt động tối ưu.

In [4]:
# Khởi tạo bộ chuẩn hóa Z-score
scaler = StandardScaler()

# Tiến hành chuẩn hóa các cột số gốc
numerical_features = ['Age', 'Annual Income', 'Spending Score']
scaled_features = [f"{col}_scaled" for col in numerical_features]

df[scaled_features] = scaler.fit_transform(df[numerical_features])

print("Dữ liệu sau khi chuẩn hóa Z-score (5 dòng đầu):")
df[scaled_features].head()

Dữ liệu sau khi chuẩn hóa Z-score (5 dòng đầu):


,Age_scaled,Annual Income_scaled,Spending Score_scaled
0,-1.145516,0.798813,1.337059
1,0.180335,1.442076,1.545929
2,0.369742,-0.743208,0.884507
3,-1.476979,-1.038953,0.223086
4,-1.429627,0.842317,0.884507


## Bước 5. Tạo Đặc Trưng Mới (Feature Creation / Generation)

Tạo đặc trưng tỷ lệ điểm chi tiêu trên thu nhập (`Spending_to_Income_Ratio`) để nắm bắt tốt hơn mối tương quan giữa hành vi chi tiêu và năng lực tài chính.

In [5]:
# 1. Tạo biến mới biểu thị tỷ lệ điểm chi tiêu trên thu nhập
df['Spending_to_Income_Ratio'] = df['Spending Score'] / df['Annual Income']

# 2. Chuẩn hóa biến mới tạo
df['Spending_to_Income_Ratio_scaled'] = scaler.fit_transform(df[['Spending_to_Income_Ratio']])

print("Thống kê mô tả của Spending_to_Income_Ratio:")
print(df['Spending_to_Income_Ratio'].describe())
df.head()

Thống kê mô tả của Spending_to_Income_Ratio:
count    15079.000000
mean         0.000651
std          0.000659
min          0.000005
25%          0.000238
50%          0.000462
75%          0.000797
max          0.004867
Name: Spending_to_Income_Ratio, dtype: float64


,Customer ID,Age,Gender,Annual Income,Spending Score,Gender_encoded,Age_scaled,Annual Income_scaled,Spending Score_scaled,Spending_to_Income_Ratio,Spending_to_Income_Ratio_scaled
0,d410ea53-6661-42a9-ad3a-f554b05fd2a7,30,Male,151479,89,1,-1.145516,0.798813,1.337059,0.000588,-0.095878
1,1770b26f-493f-46b6-837f-4237fb5a314e,58,Female,185088,95,0,0.180335,1.442076,1.545929,0.000513,-0.208511
2,e81aa8eb-1767-4b77-87ce-1620dc732c5e,62,Female,70912,76,0,0.369742,-0.743208,0.884507,0.001072,0.638431
3,9795712a-ad19-47bf-8886-4f997d6046e3,23,Male,55460,57,1,-1.476979,-1.038953,0.223086,0.001028,0.571730
4,64139426-2226-4cd6-bf09-91bce4b4db5e,24,Male,153752,76,1,-1.429627,0.842317,0.884507,0.000494,-0.237274


## Bước 6. Lựa Chọn Đặc Trưng & Lưu Dữ Liệu (Feature Selection & Saving)

Lựa chọn các đặc trưng quan trạng cho phân cụm, lưu tập dữ liệu và hiển thị bảng tổng hợp.

In [6]:
# 1. Chọn danh sách đặc trưng cuối cùng cho mô hình phân cụm
final_features = [
    'Customer ID',
    'Gender_encoded',
    'Age_scaled',
    'Annual Income_scaled',
    'Spending Score_scaled',
    'Spending_to_Income_Ratio_scaled'
]

preprocessed_df = df[final_features]

# Tạo thư mục ready_train nếu chưa tồn tại
output_dir = '../data/ready_train'
os.makedirs(output_dir, exist_ok=True)

# Lưu dữ liệu đã tiền xử lý thành công sang dạng CSV
output_path = os.path.join(output_dir, 'shopping_mall_preprocessed.csv')
preprocessed_df.to_csv(output_path, index=False)
print(f"Dữ liệu huấn luyện đã được lưu thành công tại: {os.path.abspath(output_path)}")

# 2. Tạo bảng DataFrame tổng hợp các đặc trưng đã xử lý
summary_features = [
    {'Đặc trưng': 'Customer ID', 'Kiểu gốc': 'str', 'Kiểu biến đổi': 'str', 'Phép biến đổi': 'Giữ nguyên', 'Vai trò trong mô hình': 'Định danh (Key)'},
    {'Đặc trưng': 'Gender_encoded', 'Kiểu gốc': 'str', 'Kiểu biến đổi': 'int', 'Phép biến đổi': 'Binary Mapping (Male: 1, Female: 0)', 'Vai trò trong mô hình': 'Đặc trưng đầu vào (Feature)'},
    {'Đặc trưng': 'Age_scaled', 'Kiểu gốc': 'int64', 'Kiểu biến đổi': 'float64', 'Phép biến đổi': 'StandardScaler', 'Vai trò trong mô hình': 'Đặc trưng đầu vào (Feature)'},
    {'Đặc trưng': 'Annual Income_scaled', 'Kiểu gốc': 'int64', 'Kiểu biến đổi': 'float64', 'Phép biến đổi': 'StandardScaler', 'Vai trò trong mô hình': 'Đặc trưng đầu vào (Feature)'},
    {'Đặc trưng': 'Spending Score_scaled', 'Kiểu gốc': 'int64', 'Kiểu biến đổi': 'float64', 'Phép biến đổi': 'StandardScaler', 'Vai trò trong mô hình': 'Đặc trưng đầu vào (Feature)'},
    {'Đặc trưng': 'Spending_to_Income_Ratio_scaled', 'Kiểu gốc': 'N/A (Tạo mới)', 'Kiểu biến đổi': 'float64', 'Phép biến đổi': 'Ratio & StandardScaler', 'Vai trò trong mô hình': 'Đặc trưng đầu vào (Feature)'}
]

df_summary_fe = pd.DataFrame(summary_features)
df_summary_fe

Dữ liệu huấn luyện đã được lưu thành công tại: C:\Users\Win 11\Documents\ML\ML_proj_26\lab3\data\ready_train\shopping_mall_preprocessed.csv


,Đặc trưng,Kiểu gốc,Kiểu biến đổi,Phép biến đổi,Vai trò trong mô hình
0,Customer ID,str,str,Giữ nguyên,Định danh (Key)
1,Gender_encoded,str,int,"Binary Mapping (Male: 1, Female: 0)",Đặc trưng đầu vào (Feature)
2,Age_scaled,int64,float64,StandardScaler,Đặc trưng đầu vào (Feature)
3,Annual Income_scaled,int64,float64,StandardScaler,Đặc trưng đầu vào (Feature)
4,Spending Score_scaled,int64,float64,StandardScaler,Đặc trưng đầu vào (Feature)
5,Spending_to_Income_Ratio_scaled,N/A (Tạo mới),float64,Ratio & StandardScaler,Đặc trưng đầu vào (Feature)
